### LangSmith Evaluation — Retail Supervisor Agent

In [ ]:
# %% [markdown]
# # LangSmith Evaluation — Retail Supervisor Agent (Optimized)
# Base and evaluator model: openai/gpt-oss-120b

# %%
import os
import json
import time
import sqlite3
import threading
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIError
from tavily import TavilyClient
from langsmith import Client, traceable
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from typing import List, Dict, Any, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGCHAIN_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is not set.")
if not LANGSMITH_API_KEY:
    raise ValueError("LANGSMITH_API_KEY is not set.")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

ls_client = Client(api_key=LANGSMITH_API_KEY)

BASE_MODEL = "openai/gpt-oss-120b"
EVALUATOR_MODEL = "openai/gpt-oss-120b"
DATASET_NAME = "Retail_Agent_Tool_Calling__Benchmark"

TARGET_TOKENS_PER_MINUTE = 5000
MIN_REQUEST_GAP_SECONDS = 1.0
MAX_RETRIES = 2
MAX_TOOL_CALLS = 3  # Hard limit on tool invocations per user request

_rate_lock = threading.Lock()
_rate_window_start = time.monotonic()
_rate_window_tokens = 0

def _estimate_messages_tokens(messages):
    text = json.dumps(messages, ensure_ascii=False)
    return max(1, len(text) // 4)

def _throttle_before_request(estimated_tokens):
    global _rate_window_start, _rate_window_tokens

    with _rate_lock:
        now = time.monotonic()
        elapsed = now - _rate_window_start

        if elapsed >= 60:
            _rate_window_start = now
            _rate_window_tokens = 0
            elapsed = 0

        if _rate_window_tokens + estimated_tokens > TARGET_TOKENS_PER_MINUTE:
            sleep_for = max(0.0, 60.0 - elapsed)
            print(f"⏳ Local token budget reached; sleeping {sleep_for:.1f}s...")
            time.sleep(sleep_for)
            _rate_window_start = time.monotonic()
            _rate_window_tokens = 0

        _rate_window_tokens += estimated_tokens

    time.sleep(MIN_REQUEST_GAP_SECONDS)

@traceable(name="supervisor_llm", run_type="llm")
def supervisor_llm(messages):
    _throttle_before_request(_estimate_messages_tokens(messages) + 700)

    for attempt in range(MAX_RETRIES + 1):
        try:
            return client.chat.completions.create(
                model=BASE_MODEL,
                messages=messages,
                tools=tools_schema,
                tool_choice="auto",
                max_tokens=700,
            )
        except RateLimitError:
            if attempt >= MAX_RETRIES:
                raise
            delay = min(30.0, 5.0 * (2 ** attempt))
            print(f"⚠️ Supervisor 429; retrying once after {delay:.1f}s...")
            time.sleep(delay)
        except APIError:
            raise

@traceable(name="judge_llm", run_type="llm")
def judge_llm(messages):
    _throttle_before_request(_estimate_messages_tokens(messages) + 180)

    for attempt in range(MAX_RETRIES + 1):
        try:
            return client.chat.completions.create(
                model=EVALUATOR_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=180,
            )
        except RateLimitError:
            if attempt >= MAX_RETRIES:
                raise
            delay = min(30.0, 5.0 * (2 ** attempt))
            print(f"⚠️ Judge 429; retrying once after {delay:.1f}s...")
            time.sleep(delay)
        except APIError:
            raise

print(f"Base model: {BASE_MODEL}")
print(f"Evaluator model: {EVALUATOR_MODEL}")

Base model: meta-llama/llama-3.3-70b-instruct
Evaluator model: openai/gpt-oss-120b


### Benchmark Dataset Creation

In [2]:
# %% [markdown]
# ### Benchmark Dataset Creation (Simplified & Concise Prompts)

# %%
benchmark_data = [
    {
        'inputs': {'question': "What is the restocking fee percentage for opened laptops returned within 14 days under OmniTech policy?"},
        'outputs': {'ground_truth': "15%"}
    },
    {
        'inputs': {'question': "Which product is the most expensive in the products table?"},
        'outputs': {'ground_truth': "Flagship Laptop Pro X15"}
    },
    {
        'inputs': {'question': "What is the sum of product prices for product IDs 1 through 5 in SQLite?"},
        'outputs': {'ground_truth': "1989.46"}
    },
    {
        'inputs': {'question': "What is the shipping cost for TVs over 65 inches under OmniTech policy?"},
        'outputs': {'ground_truth': "$89.00"}
    },
    {
        'inputs': {'question': "What is the primary penalty for reselling discounted OmniTech products?"},
        'outputs': {'ground_truth': "Immediate contract termination and legal action"}
    },
    {
        'inputs': {'question': "What is the total price for 2 units of each non-Accessories product for product IDs 1 to 8?"},
        'outputs': {'ground_truth': "3958.94"}
    },
    {
        'inputs': {'question': "What is the B2B bulk discount percentage for orders of 250 units?"},
        'outputs': {'ground_truth': "18%"}
    },
    {
        'inputs': {'question': "What is the price of product ID 3 (Gaming Monitor G27) in USD?"},
        'outputs': {'ground_truth': "$349.50"}
    },
    {
        'inputs': {'question': "Are standalone laptop spare batteries permitted on express air shipping?"},
        'outputs': {'ground_truth': "No"}
    },
    {
        'inputs': {'question': "What is the dollar deductible amount for OmniCare liquid damage claims?"},
        'outputs': {'ground_truth': "$50.00"}
    },
    {
        'inputs': {'question': "How many customer email addresses end in .org in the customers table?"},
        'outputs': {'ground_truth': "12"}
    },
    {
        'inputs': {'question': "In what form are Bitcoin purchase refunds issued under OmniTech policy?"},
        'outputs': {'ground_truth': "OmniTech Store Credit"}
    },
    {
        'inputs': {'question': "What is the trade-in value reduction percentage for Grade B devices?"},
        'outputs': {'ground_truth': "25%"}
    },
    {
        'inputs': {'question': "Is a government photo ID required to pick up Express In-Store orders over $1,000?"},
        'outputs': {'ground_truth': "Yes"}
    },
    {
        'inputs': {'question': "What is the final total cost for 60 Noise Cancelling Earbuds with 12% discount and 7.25% California tax?"},
        'outputs': {'ground_truth': "8493.63"}
    }
]

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    print(f"✅ Reusing existing LangSmith dataset: {DATASET_NAME}")
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="15 simplified QA pairs for low-recursion supervisor evaluation.",
    )
    for item in benchmark_data:
        ls_client.create_example(
            inputs=item["inputs"],
            outputs=item["outputs"],
            dataset_id=dataset.id,
        )
    print(f"✅ Created dataset with {len(benchmark_data)} examples.")

✅ Reusing existing LangSmith dataset: Retail_Agent_Tool_Calling__Benchmark


### Tool Execution Logic

In [3]:
# %% [markdown]
# ### Tool Execution Logic

# %%
CHROMA_DIR = os.path.join("data", "chroma_db")
SQLITE_DB_PATH = os.path.join("data", "company.db")

print("⏳ Loading local HuggingFace embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(CHROMA_DIR):
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
    )
    print(f"✅ Loaded ChromaDB: {CHROMA_DIR}")
else:
    vectorstore = None
    print(f"⚠️ ChromaDB not found: {CHROMA_DIR}")

@traceable(name="execute_tavily_search")
def execute_tavily_search(query: str, max_results: int = 3) -> str:
    try:
        response = TavilyClient(api_key=TAVILY_API_KEY).search(
            query=query,
            max_results=max_results,
            search_depth="basic",
        )
        results = response.get("results", [])
        if not results:
            return "NO_EVIDENCE: Tavily returned zero search results."
        return "\n".join(
            f"- Title: {r.get('title')}\n  URL: {r.get('url')}\n  Snippet: {r.get('content')}"
            for r in results
        )
    except Exception as e:
        return f"TOOL_ERROR (Tavily): {e}"

@traceable(name="query_knowledge_base")
def query_knowledge_base(query: str, top_k: int = 3) -> str:
    if vectorstore is None:
        return f"TOOL_ERROR: ChromaDB is not initialized: {CHROMA_DIR}"
    try:
        docs = vectorstore.similarity_search(query, k=top_k)
        if not docs:
            return f"NO_EVIDENCE: No KB entries found for '{query}'."
        return "[Knowledge Base RAG Results]\n" + "\n".join(
            f"- {doc.page_content}" for doc in docs
        )
    except Exception as e:
        return f"TOOL_ERROR (ChromaDB): {e}"

@traceable(name="query_sales_db")
def query_sales_db(sql_query: str) -> str:
    if not sql_query.strip().lower().startswith("select"):
        return "TOOL_ERROR: Only read-only SELECT queries are allowed."
    if not os.path.exists(SQLITE_DB_PATH):
        return f"TOOL_ERROR: SQLite database not found: {SQLITE_DB_PATH}"

    try:
        with sqlite3.connect(SQLITE_DB_PATH) as conn:
            cursor = conn.cursor()
            cursor.execute(sql_query)
            columns = [d[0] for d in cursor.description]
            rows = cursor.fetchall()

        if not rows:
            return "[SQLite Execution Result]: 0 rows."

        return f"[SQLite Execution Result]: {[dict(zip(columns, row)) for row in rows]}"
    except Exception as e:
        return f"TOOL_ERROR (SQLite): {e}"

@traceable(name="call_tool_by_name")
def call_tool_by_name(tool_name: str, tool_args: dict) -> str:
    if tool_name == "execute_tavily_search":
        return execute_tavily_search(tool_args["query"])
    if tool_name == "query_knowledge_base":
        return query_knowledge_base(tool_args["query"])
    if tool_name == "query_sales_db":
        return query_sales_db(tool_args["sql_query"])
    return f"TOOL_ERROR: Tool '{tool_name}' is not recognized."

⏳ Loading local HuggingFace embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Loaded ChromaDB: data\chroma_db


### Tool Schemas, State Guardrails, and LangGraph Workflow

In [4]:
# %% [markdown]
# ### Tool Schemas, State Guardrails, and LangGraph Workflow

# %%
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "execute_tavily_search",
            "description": "Search the live web using Tavily for market facts, competitor prices, and external news.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_knowledge_base",
            "description": "Search internal vector database for company policies, warranties, manuals, and FAQs.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Semantic retrieval query."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_sales_db",
            "description": (
                "Execute read-only SQL SELECT queries on SQLite company.db.\n"
                "Schema:\n"
                "- customers(customer_id, name, email, phone, address, created_at)\n"
                "- products(product_id, name, category, price, stock_quantity)\n"
                "- orders(order_id, customer_id, order_date, total_amount, status)\n"
                "- order_items(order_item_id, order_id, product_id, quantity, unit_price)"
            ),
            "parameters": {
                "type": "object",
                "properties": {"sql_query": {"type": "string", "description": "Valid SQL SELECT query."}},
                "required": ["sql_query"],
            },
        },
    },
]

SUPERVISOR_SYSTEM_PROMPT = """You are a Retail Intelligence Supervisor Agent with access to 3 tools:
1. query_sales_db: SQLite database for sales numbers, products, orders.
2. query_knowledge_base: Vector search across policies, manuals, FAQs.
3. execute_tavily_search: Live web search for external market facts.

STRICT TOOL BUDGET RULES:
1. MAXIMUM 3 TOOL CALLS TOTAL: Most queries require ONLY 1 or 2 tool calls.
2. PARALLEL BATCHING: If you need data from multiple tools, execute them in a SINGLE parallel turn on your first step.
3. NO RETRY LOOPS: Do NOT perform trial-and-error follow-up searches.
4. SYNTHESIZE AND FINISH: Base answers ONLY on retrieved evidence. If evidence is insufficient, state: "I don't know based on available evidence."
"""

def add_messages_reducer(left, right):
    return left + right

# State includes tool_call_count guardrail
class AgentState(TypedDict):
    messages: Annotated[List[Dict[str, Any]], add_messages_reducer]
    tool_call_count: int

def compact_messages(messages):
    """Summarizes prior tool evidence to keep token window lightweight."""
    if len(messages) <= 4:
        return messages

    evidence = []
    for msg in messages[2:-1]:
        if msg.get("role") == "tool":
            content = str(msg.get("content", ""))
            evidence.append(f"[{msg.get('name', 'Tool')}]: {content[:250]}...")

    return [
        messages[0],
        messages[1],
        {
            "role": "user",
            "content": "[PRIOR RETRIEVED EVIDENCE]\n" + ("\n".join(evidence) or "None."),
        },
    ] + messages[-1:]

@traceable(name="supervisor_agent_node")
def call_supervisor_agent_node(state: AgentState):
    messages = compact_messages(state["messages"])
    response = supervisor_llm(messages)
    message = response.choices[0].message

    assistant_msg = {
        "role": "assistant",
        "content": message.content or "",
    }

    if message.tool_calls:
        assistant_msg["tool_calls"] = [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in message.tool_calls
        ]

    return {"messages": [assistant_msg]}

@traceable(name="execute_tools_node")
def execute_tools_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_calls = last_message.get("tool_calls", [])
    
    # Track executed tool count
    current_count = state.get("tool_call_count", 0) + len(tool_calls)
    tool_responses = []

    for tool_call in tool_calls:
        tool_name = tool_call["function"]["name"]
        tool_args = json.loads(tool_call["function"]["arguments"])
        print(f"🔄 Executing {tool_name}: {tool_args}")

        tool_output = call_tool_by_name(tool_name, tool_args)

        tool_responses.append({
            "tool_call_id": tool_call["id"],
            "role": "tool",
            "name": tool_name,
            "content": str(tool_output),
        })

    return {
        "messages": tool_responses,
        "tool_call_count": current_count
    }

def route_after_agent(state: AgentState):
    """Enforces a hard ceiling of MAX_TOOL_CALLS (3) or terminates if no tools called."""
    last_message = state["messages"][-1]
    tool_calls = last_message.get("tool_calls", [])
    total_calls = state.get("tool_call_count", 0)

    # Hard cap check: Force end if tools >= 3 OR no tool requested
    if total_calls >= MAX_TOOL_CALLS or not tool_calls:
        return END

    return "tools"

# Construct graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_supervisor_agent_node)
workflow.add_node("tools", execute_tools_node)
workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", END: END},
)
workflow.add_edge("tools", "agent")
app = workflow.compile()

@traceable(name="run_supervisor_agent")
def run_supervisor_agent(inputs: dict) -> dict:
    time.sleep(2.0)  # Throttling gap
    
    question = inputs["question"]
    
    try:
        final_state = app.invoke(
            {
                "messages": [
                    {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
                    {"role": "user", "content": question},
                ],
                "tool_call_count": 0
            },
            config={"recursion_limit": 15}, # Higher limit allows multi-hop steps cleanly
        )
        answer = final_state["messages"][-1].get("content", "")
        message_count = len(final_state["messages"])
    except Exception as e:
        answer = f"Execution error: {e}"
        message_count = 0

    return {
        "answer": answer,
        "message_count": message_count,
    }

print("✅ Guardrailed supervisor graph compiled successfully.")

✅ Guardrailed supervisor graph compiled successfully.


### LLM Evaluator Node & Defensive Parsing

In [5]:
# %% [markdown]
# ### LLM Evaluator Node & Defensive Parsing (Fixed)

# %%
import re
import json

EVALUATOR_PROMPT = """You are an expert evaluator.

Grade the candidate answer against the ground truth using:
1. Correctness: facts, numbers, policy details, and calculations.
2. Completeness: every requested sub-question is answered.
3. Evidence discipline: unsupported claims should not receive full credit.

Return ONLY valid JSON:
{{"score": 0.0, "reason": "brief reason"}}

Score guide:
1.0 = fully correct and complete
0.75 = mostly correct, minor omission
0.5 = materially incomplete or partly incorrect
0.0 = wrong/unusable

Question:
{question}

Ground truth:
{ground_truth}

Candidate answer:
{candidate}
"""

def parse_judge_json(content: str):
    """Robustly parses score and reason from LLM judge output to prevent KeyError."""
    if not content:
        return 0.0, "Empty judge output"

    # Clean markdown code fences and whitespace
    cleaned = content.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        data = json.loads(cleaned)
        
        # Strip literal outer quotes or whitespace from dictionary keys if model output '"score"'
        normalized_data = {
            str(k).replace('"', '').replace("'", '').strip(): v 
            for k, v in data.items()
        }
        
        score = float(normalized_data.get("score", 0.0))
        reason = str(normalized_data.get("reason", "No reason provided"))
        return max(0.0, min(1.0, score)), reason[:500]

    except Exception:
        # Fallback regex extraction if JSON structure is malformed
        score_match = re.search(r'["\']?score["\']?\s*:\s*([0-9]+(?:\.[0-9]+)?)', content, re.IGNORECASE)
        reason_match = re.search(r'["\']?reason["\']?\s*:\s*["\']([^"\']+)["\']', content, re.IGNORECASE)

        score = float(score_match.group(1)) if score_match else 0.0
        reason = reason_match.group(1) if reason_match else "Extracted via regex fallback"

        return max(0.0, min(1.0, score)), reason[:500]

@traceable(name="evaluate_llm_judge")
def evaluate_llm_judge(inputs: dict, outputs: dict, reference_outputs: dict):
    # Safe field extraction using .get() prevents KeyError on input dicts
    question = inputs.get("question", "") if isinstance(inputs, dict) else ""
    ground_truth = reference_outputs.get("ground_truth", "") if isinstance(reference_outputs, dict) else ""
    candidate = outputs.get("answer", "") if isinstance(outputs, dict) else ""

    prompt = EVALUATOR_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        candidate=candidate,
    )

    try:
        response = judge_llm([{"role": "user", "content": prompt}])
        score, reason = parse_judge_json(response.choices[0].message.content)
    except Exception as e:
        score = 0.0
        reason = f"Evaluation failed safely: {type(e).__name__} - {e}"

    return {
        "key": "correctness_score",
        "score": score,
        "comment": reason,
    }

### Evaluation Runner Execution

In [ ]:
# %% [markdown]
# ### Evaluation Runner Execution

# %%
from langsmith.evaluation import evaluate

results = evaluate(
    run_supervisor_agent,
    data=DATASET_NAME,
    evaluators=[evaluate_llm_judge],
    experiment_prefix="GPT-OSS-120B-Supervisor-Eval",
    max_concurrency=1,
    num_repetitions=1,
)

print("✅ Evaluation completed.")
print(results)

View the evaluation results for experiment: 'GPT-OSS-120B-Supervisor-Eval-c071b90a' at:
https://apac.smith.langchain.com/o/de69f496-902d-4ca7-b209-fd22b9f10dad/datasets/ec70e37e-5726-49af-ac2c-75172a941db2/compare?selectedSessions=b7f8457e-934f-4012-9b00-6b4a41e64875




0it [00:00, ?it/s]

⏳ Local token budget reached; sleeping 1.1s...
⏳ Local token budget reached; sleeping 44.0s...
